#3.6 The Full Transformer LM

##(a)
Consider GPT-2 XL, which has the following configuration:

vocab_size : 50,257

context_length : 1,024

num_layers : 48

d_model : 1,600

num_heads : 25

d_ff : 6,400

Suppose we constructed our model using this configuration. How many trainable parameters
would our model have? Assuming each parameter is represented using single-precision floating
point, how much memory is required to just load this model?
Deliverable: A one-to-two sentence response.


trainable components are:

token embedding: (vocab_size, d_model)

transformer block * num_layers:

- RMS norm: (d_model, )
- Multi-Head self-attention (across all heads):
  - Wq: (num_heads * (d_model / num_heads), d_model)
  - Wk: (num_heads * (d_model / num_heads), d_model)
  - Wv: (num_heads * (d_model / num_heads), d_model)
  - Wo: (d_model, d_model)
- RMS norm: (d_model, )
- FFN (SwiGLU):
  - W1: (d_model, d_ff)
  - W2: (d_ff, d_model)
  - W3: (d_model, d_ff)

final RMS norm (gain vector): (d_model, )

final linear activation: (vocab_size, d_model)


In [5]:
# Answer:

vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 6400

trainable_params = vocab_size * d_model + num_layers * (d_model*2 + d_model*d_model*4 + d_model*d_ff*3) + d_model + vocab_size * d_model
print(trainable_params)

2127057600


## (b)
Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped
model. How many FLOPs do these matrix multiplies require in total? Assume that our input
sequence has context_length tokens.
Deliverable: A list of matrix multiplies (with descriptions), and the total number of FLOPs
required.

Computations happen in:

0. pre-compute rope: **context_length * (2*d_model)**

1. token embedding:
- input: (context_length, )
- look-up in embedding matrix
- output: (context_length, d_model)

2. transformer block * num_layers:
  1. RMS norm:
  - input: (context_length, d_model)
  - RMS computation: **context_length * 4 * d_model**
  - output: (context_length, d_model)
  2. Multihead SelfAttention:
  - input: (context_length, d_model)
  - projection (get Q, K, V): **3 * (2 * context_length * d_model^2)**
  - rope on Q, K: **2 * (context_length * 3 * d_model)**
  - Multihead attention: **num_heads * [2 * context_length^2 * (d_model/num_heads) + 4*context_length^2 + 2 * context_length^2 * (d_model/num_heads)]**
  - Output projection: **2 * context_len * d_model^2**
  - Residual layer: **context_len * d_model**
  - output: (context_length, d_model)
  3. RMS norm:
  - input: (context_length, d_model)
  - RMS computation: **context_length * 4 * d_model**
  - output: (context_length, d_model)
  4. FFN (SwiGLU):
  - input: (context_length, d_model)
  - W1 * x: **2 * d_ff * d_model * context_len**
  - SiLU(W1 * x): **4 * context_len * d_ff**
  - W3 * x: **2 * d_ff * d_model * context_len**
  - sum: **d_ff * context_len**
  - forward: **2 * context_len * d_ff * d_model**
  - Residual layer: **context_len * d_model**
  - output: (context_length, d_model)

3. final RMS norm:
  - input: (context_length, d_model)
  - RMS computation: **context_length * 4 * d_model**
  - output: (context_length, d_model)

4. Linear activation:
  - input: (context_length, d_model)
  - forward: **2 * context_len * d_model * vocab_size**
  - output: (context_length, vocab_size)

5. Softmax output:
  - input: (context_length, vocab_size)
  - softmax: **3 * context_length * vocab_size**
  - output: (context_length, vocab_size)

In [16]:
# Answer: 4.52e12
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 6400


pre_compute_rope = context_length * (2*d_model)
RMS = context_length * 5 * d_model
MHA = int(sum([
    3 * (2 * context_length * d_model**2),
    2 * (context_length * 3 * d_model),
    num_heads * (2 * context_length**2 * (d_model/num_heads) + 4*context_length**2 + 2 * context_length**2 * (d_model/num_heads)),
    2 * context_length * d_model**2,
    context_length * d_model
]))
FFN = int(sum([
    2 * context_length * d_ff * d_model,
    4 * context_length * d_ff,
    2 * context_length * d_ff * d_model,
    d_ff * context_length,
    2 * context_length * d_ff * d_model,
    context_length * d_model
]))
Linear = 2 * context_length * d_model * vocab_size
softmax = 3 * context_length * vocab_size

total_FLOPs = num_layers * (MHA + FFN + 2 * RMS) + RMS + Linear + softmax
print(total_FLOPs)

4521520712704


##(c)
Based on your analysis above, which parts of the model require the most FLOPs?

In [17]:
steps = {'precompute rope':pre_compute_rope, 'Transformer':2 * RMS + MHA + FFN, 'Final RMS':RMS, 'Linear Output':Linear + softmax}
print(sorted(steps.items(), reverse = True, key=lambda x:x[1]))

[('Linear Output', 164836527104), ('Transformer', 90764083200), ('Final RMS', 8192000), ('precompute rope', 3276800)]


##(d)
Repeat your analysis with GPT-2 small (12 layers, 768 d_model, 12 heads), GPT-2 medium (24
layers, 1024 d_model, 16 heads), and GPT-2 large (36 layers, 1280 d_model, 20 heads). As the
model size increases, which parts of the Transformer LM take up proportionally more or less of
the total FLOPs?

In [37]:
# Answer:

def compute_params(dic):
    vocab_size = dic['vocab_size']
    context_length = dic['context_length']
    num_layers = dic['num_layers']
    d_model = dic['d_model']
    num_heads = dic['num_heads']
    d_ff = dic['d_ff']
    return vocab_size * d_model + num_layers * (d_model*2 + d_model*d_model*4 + d_model*d_ff*3) + d_model + vocab_size * d_model

def compute_flops(dic):
    vocab_size = dic['vocab_size']
    context_length = dic['context_length']
    num_layers = dic['num_layers']
    d_model = dic['d_model']
    num_heads = dic['num_heads']
    d_ff = dic['d_ff']

    pre_compute_rope = context_length * (2*d_model)
    RMS = context_length * 5 * d_model
    MHA = int(sum([
        3 * (2 * context_length * d_model**2),
        2 * (context_length * 3 * d_model),
        num_heads * (2 * context_length**2 * (d_model/num_heads) + 4*context_length**2 + 2 * context_length**2 * (d_model/num_heads)),
        2 * context_length * d_model**2,
        context_length * d_model
    ]))
    FFN = int(sum([
        2 * context_length * d_ff * d_model,
        4 * context_length * d_ff,
        2 * context_length * d_ff * d_model,
        d_ff * context_length,
        2 * context_length * d_ff * d_model,
        context_length * d_model
    ]))
    Linear = 2 * context_length * d_model * vocab_size
    softmax = 3 * context_length * vocab_size

    total_FLOPs = num_layers * (MHA + FFN + 2 * RMS) + RMS + Linear + softmax
    return total_FLOPs, {'precompute rope':pre_compute_rope,
                         'Transformer Total':num_layers * (2 * RMS + MHA + FFN),
                         'Each Transformer':2 * RMS + MHA + FFN,
                         'Final RMS':RMS,
                         'Linear':Linear,
                         'Softmax':softmax}



GPT2XL_dic = {'vocab_size':50257, 'context_length':1024, 'num_layers':48, 'd_model':1600, 'num_heads':25, 'd_ff':6400}
GPT2small_dic = {'vocab_size':50257, 'context_length':1024, 'num_layers':12, 'd_model':768, 'num_heads':12, 'd_ff':6400}
GPT2medium_dic = {'vocab_size':50257, 'context_length':1024, 'num_layers':24, 'd_model':1024, 'num_heads':16, 'd_ff':6400}
GPT2large_dic = {'vocab_size':50257, 'context_length':1024, 'num_layers':36, 'd_model':1280, 'num_heads':20, 'd_ff':6400}

total_XL, steps_XL = compute_flops(GPT2XL_dic)
param_XL = compute_params(GPT2XL_dic)
for dic in [GPT2small_dic, GPT2medium_dic, GPT2large_dic][::-1]:
    print('\ndownsizing...\n')
    total_FLOPs, steps = compute_flops(dic)
    print(f'A. total param ratio: {compute_params(dic)/param_XL}')
    print(f'B. total flops ratio: {total_FLOPs/total_XL:.4f}')
    for ind, (k, v) in enumerate(steps.items()):
        print(f'   {ind}: {k}')
        print(f'      composition: {v/total_FLOPs:.4f}')
        print(f'      flops ratio: {v/steps_XL[k]:.4f}')
        print(f'      saving ratio: {(steps_XL[k] - v)/(total_XL - total_FLOPs):.4f}')





downsizing...

A. total param ratio: 0.5873921608892961
B. total flops ratio: 0.5806
   0: precompute rope
      composition: 0.0000
      flops ratio: 0.8000
      saving ratio: 0.0000
   1: Transformer Total
      composition: 0.9498
      flops ratio: 0.5723
      saving ratio: 0.9826
   2: Each Transformer
      composition: 0.0264
      flops ratio: 0.7631
      saving ratio: 0.0113
   3: Final RMS
      composition: 0.0000
      flops ratio: 0.8000
      saving ratio: 0.0000
   4: Linear
      composition: 0.0502
      flops ratio: 0.8000
      saving ratio: 0.0174
   5: Softmax
      composition: 0.0001
      flops ratio: 1.0000
      saving ratio: 0.0000

downsizing...

A. total param ratio: 0.3175743844454424
B. total flops ratio: 0.3061
   0: precompute rope
      composition: 0.0000
      flops ratio: 0.6400
      saving ratio: 0.0000
   1: Transformer Total
      composition: 0.9237
      flops ratio: 0.2934
      saving ratio: 0.9811
   2: Each Transformer
      compositi

##(e)

Take GPT-2 XL and increase the context length to 16,384. How does the total FLOPs for one
forward pass change? How do the relative contribution of FLOPs of the model components
change?

In [42]:
# Answer: The composition of Flops from Transformers increased from 0.9635 to 0.9825
GPT2XL_dic = {'vocab_size':50257, 'context_length':1024, 'num_layers':48, 'd_model':1600, 'num_heads':25, 'd_ff':6400}
GPT2XXL_dic = {'vocab_size':50257, 'context_length':16384, 'num_layers':48, 'd_model':1600, 'num_heads':25, 'd_ff':6400}

total_XXL, steps_XXL = compute_flops(GPT2XXL_dic)
param_XXL = compute_params(GPT2XXL_dic)
for dic in [GPT2XXL_dic, GPT2XL_dic]:
    print('\ndownsizing...\n')
    total_FLOPs, steps = compute_flops(dic)
    print(f'A. total param ratio: {compute_params(dic)/param_XXL}')
    print(f'B. total flops ratio: {total_FLOPs/total_XXL:.4f}')
    for ind, (k, v) in enumerate(steps.items()):
        print(f'   {ind}: {k}')
        print(f'      composition: {v/total_FLOPs:.4f}')
        print(f'      flops ratio: {v/steps_XXL[k]:.4f}')
        if total_XXL - total_FLOPs > 0:
            print(f'      saving ratio: {(steps_XXL[k] - v)/(total_XXL - total_FLOPs):.4f}')


downsizing...

A. total param ratio: 1.0
B. total flops ratio: 1.0000
   0: precompute rope
      composition: 0.0000
      flops ratio: 1.0000
   1: Transformer Total
      composition: 0.9825
      flops ratio: 1.0000
   2: Each Transformer
      composition: 0.0205
      flops ratio: 1.0000
   3: Final RMS
      composition: 0.0000
      flops ratio: 1.0000
   4: Linear
      composition: 0.0175
      flops ratio: 1.0000
   5: Softmax
      composition: 0.0000
      flops ratio: 1.0000

downsizing...

A. total param ratio: 1.0
B. total flops ratio: 0.0300
   0: precompute rope
      composition: 0.0000
      flops ratio: 0.0625
      saving ratio: 0.0000
   1: Transformer Total
      composition: 0.9635
      flops ratio: 0.0294
      saving ratio: 0.9831
   2: Each Transformer
      composition: 0.0201
      flops ratio: 0.0294
      saving ratio: 0.0205
   3: Final RMS
      composition: 0.0000
      flops ratio: 0.0625
      saving ratio: 0.0000
   4: Linear
      composition: 0